# D1.8 · Threat intel sub-lane

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.7 · Drift monitoring](https://spbreed.github.io/cyber-commons/lessons/D1.7.html)**.

| | |
|---|---|
| Tools used | MISP, OpenCTI, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Build a synthesis loop that must cite or abstain.

**Why a security engineer needs it.** Unsourced confidence in synthesis loops. The control it builds is: provenance discipline; refuse claims without a source.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Two intel questions, not one: how adversaries are using AI, and who is coming for the AI you run. Most programmes track the first because it is written about, and the second is the one that reaches your estate.

> **At CyberTravels.** Two intel questions for CyberTravels: how adversaries use agents, and who is coming for CyberTravels. The second is the one that reaches the booking API.

## 2 · The framework

```
   two intel questions, only one of which is well covered

   how adversaries use AI        who is coming for the AI you run
   +---------------------+       +-----------------------------+
   | written about a lot |       | your models, agents, MCP    |
   | mostly capability   |       | servers, eval corpora       |
   +---------------------+       +-----------------------------+
                                       the one that reaches you
```

Threat intel is judged by exactly one thing: **how many detections came out of
it.** Everything else — feed volume, report quality, briefing frequency — is
input, not outcome.

An indicator is actionable when two things are true:

- it is a **type you can match on** (a host, a hash, a specific technique with a
  concrete precondition), and
- its **confidence justifies the false-positive cost** of the rule it becomes.

A narrative about adversary trends is not intelligence you can operate. It may
be genuinely useful for planning and it should not be counted as detection
coverage, because counting it that way makes a programme look covered when it is
not.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">intel source</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">converts to a detection</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">why</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">your own incidents</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>100%</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the technique that worked against you</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">your red team (C1)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>90%</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">attack-suite results become detections directly</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">your drift monitor (D1.7)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>80%</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">baseline changes are leading indicators</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">vendor advisories</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">50%</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">useful for the supply chain (C2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">commercial feed</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">30%</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">generic indicators; little agent-specific content yet</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">The highest-converting sources are all internal. For agentic threats the intel programme is mostly a feedback loop out of C1 and D1.7, not a purchase.</div>

## 3 · The procedure, as a skill

Four of seven indicators convert; the two narratives and the low-confidence host are dropped with reasons. The skill then reports the three numbers a renewal conversation needs: converted, alerted, actioned.

In [ ]:
# skills/detection/threat-intel-to-rules/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: threat-intel-to-rules
description: >-
  Convert a threat-intel feed into detection rules, drop the indicators that
  cannot be matched or scored, and report the three numbers that say whether the
  feed earns its price. Use when subscribing to intel, or when a feed produces
  alerts nobody actions.
allowed-tools: Read, Grep, Glob
---

# Most of a feed is not matchable, and that is fine if you say so

An intel feed contains indicators, narratives and low-confidence guesses. Only
some of it converts into something a detection can match on. The useful output
is a small number of rules plus an explicit list of what was dropped and why —
and three numbers that tell you whether to renew.

## When to use this

On any intel feed, at subscription and at renewal, and when an intel-derived
rule fires and nobody knows what to do.

## Procedure

**1 — Set a confidence floor and a matchable-type list.** Hosts, hashes,
techniques. Narratives are context, not indicators; write the floor down before
reading the feed.

**2 — Convert what qualifies.** One rule per indicator, with the response
attached. An indicator with no response is not ready — the analyst will get the
alert and ask what to do.

**3 — Drop the rest with reasons.** "Narrative, not matchable" and "confidence
0.4, below floor" are both fine. An undocumented drop is indistinguishable from
an oversight.

**4 — Run the rules against real events.** Record which fired and what the
response was. A rule that has never fired is not automatically bad; a rule that
fires and produces no action is.

**5 — Report the three numbers.** Indicators converted, alerts produced, alerts
actioned. The ratio of the third to the second is what the feed is worth, and it
is the number to take to a renewal conversation.

## Output contract

```json
{
  "feed": {"indicators": 0, "confidence_floor": 0.0, "matchable_types": ["str"]},
  "rules": [{"indicator": "str", "type": "str", "response": "str"}],
  "dropped": [{"indicator": "str", "reason": "str"}],
  "firings": [{"rule": "str", "events": 0}],
  "value": {"converted": 0, "alerts": 0, "actioned": 0}
}
```

## Failure modes

- **Converting narratives.** They do not match on anything.
- **Rules with no response.** The alert lands and stops.
- **Reporting alerts rather than actions.** Alerts are cheap to produce.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/detection/threat-intel-to-rules/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/detection/threat-intel-to-rules/scripts/threat_intel_to_rules.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Convert an intel feed into rules, dropping what cannot be matched, and report the three numbers that say whether the feed is worth its price.

This is the executable half of the `threat-intel-to-rules` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
from dataclasses import dataclass

@dataclass
class Indicator:
    value: str; kind: str; source: str; confidence: float

FEED = [
 Indicator("collect.example.com", "host", "vendor-a", 0.95),
 Indicator("169.254.169.254", "host", "internal-research", 0.99),
 Indicator("a1b2c3d4e5f6", "hash", "vendor-b", 0.72),
 Indicator("pastebin.example", "host", "vendor-a", 0.55),
 Indicator("adversaries increasingly use agentic tooling", "narrative", "blog", 0.40),
 Indicator("agents reading ~/.aws/credentials", "technique", "internal-ir", 0.88),
 Indicator("threat actor GOLDEN-OTTER is targeting fintech", "narrative", "vendor-c", 0.60),
]
CONF_FLOOR = 0.70
MATCHABLE = {"host", "hash", "technique"}

def actionable(i):
    if i.kind not in MATCHABLE:
        return False, f"{i.kind} is not matchable in telemetry"
    if i.confidence < CONF_FLOOR:
        return False, f"confidence {i.confidence} below floor {CONF_FLOOR}"
    return True, "convertible to a rule"

print(f"{'indicator':46s}{'kind':11s}{'conf':>6}  verdict")
print("-" * 88)
for i in FEED:
    ok, why = actionable(i)
    print(f"{i.value[:44]:46s}{i.kind:11s}{i.confidence:>6.2f}  "
          f"{'RULE' if ok else 'drop'} — {why}")

@dataclass
class Rule:
    name: str; severity: str; match: object; response: str

def to_rules(feed):
    rules = []
    for i in feed:
        ok, _ = actionable(i)
        if not ok: continue
        sev = "critical" if i.confidence > 0.9 else "high"
        if i.kind == "host":
            m = (lambda v: (lambda e: v in e.get("target", "")))(i.value)
            resp = f"block egress, revoke the agent's token, hunt back 30d ({i.source})"
        elif i.kind == "hash":
            m = (lambda v: (lambda e: v == e.get("hash", "")))(i.value)
            resp = f"quarantine the artefact, check the supply chain ({i.source})"
        else:
            m = (lambda: (lambda e: "/.aws/" in e.get("target", "")))()
            resp = f"revoke, rotate the cloud role, audit reads ({i.source})"
        rules.append(Rule(f"intel:{i.kind}:{i.value[:26]}", sev, m, resp))
    return rules

rules = to_rules(FEED)
print(f"{len(FEED)} indicators → {len(rules)} deployable rules "
      f"({len(rules)/len(FEED):.0%} conversion)\n")
for r in rules:
    print(f"   [{r.severity:8s}] {r.name}")

EVENTS = [
 {"actor": "patch-agent", "target": "https://collect.example.com/x"},
 {"actor": "triage-agent", "target": "https://api.github.com/repos"},
 {"actor": "patch-agent", "target": "/home/app/.aws/credentials"},
 {"actor": "svc-etl", "target": "/data/export.csv"},
 {"actor": "build-agent", "hash": "a1b2c3d4e5f6"},
]
fired = [(r, e) for r in rules for e in EVENTS if r.match(e)]
print("alerts generated from the feed:")
for r, e in fired:
    print(f"   [{r.severity}] {r.name}")
    print(f"        actor={e['actor']}  → {r.response}")

ACTIONED = 2      # of those alerts, how many led to an action
print(f"\nthe three numbers that matter:")
print(f"   indicators received : {len(FEED)}")
print(f"   rules deployed      : {len(rules)}  ({len(rules)/len(FEED):.0%} of the feed)")
print(f"   alerts fired        : {len(fired)}")
print(f"   alerts actioned     : {ACTIONED}  ({ACTIONED/max(len(fired),1):.0%})")
print("\nThe third number is the one that decides whether the subscription renews.")

## What you just proved

Four of seven indicators convert to rules — the two narratives and the low-confidence host are dropped with reasons. The rules fire on three of five events with concrete responses. The three-number summary shows a 57% conversion rate and 67% of alerts actioned, and the source table ranks internal sources highest.

## Your turn

Compute your own three numbers for last quarter: indicators received, rules deployed, alerts actioned. The ratio between the first and third is the honest value of the programme.

---

**Next → [D1.9 · Detections whose subject is the agent platform](https://spbreed.github.io/cyber-commons/lessons/D1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*